In [1]:
import requests, json, time, os

OLLAMA_MODEL = "qwen3.5:4b"

def call_llm(prompt, model=OLLAMA_MODEL, temperature=0.4, max_tokens=500):
    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={"model": model, "prompt": prompt, "stream": False, "think": False,
                  "options": {"temperature": temperature, "num_predict": max_tokens}},
            timeout=90
        )
        return response.json().get("response", "").strip()
    except Exception as e:
        return f"LLM call failed: {e}"

def load_json_safe(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except:
        return {}

def load_text_safe(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return f.read()
    except:
        return ""

# Load all agent outputs into memory
context_data = {
    "resume_text": load_text_safe("../outputs/resume_text.txt"),
    "skills": load_json_safe("../outputs/skills.json"),
    "readiness": load_json_safe("../outputs/career_readiness.json"),
    "skill_gap": load_json_safe("../outputs/skill_gap.json"),
    "roadmap": load_json_safe("../outputs/roadmap.json"),
    "projects": load_json_safe("../outputs/project_recommendations.json"),
    "certifications": load_json_safe("../outputs/certifications.json"),
    "improvements": load_json_safe("../outputs/resume_improvements.json"),
    "interview": load_json_safe("../outputs/interview_questions.json"),
    "crafted_resume": load_text_safe("../outputs/crafted_resume.txt"),
    "job_matches": load_json_safe("../outputs/job_matches.json"),
}

print("Chat Orchestrator: all agent outputs loaded into context")
print(f"Loaded {sum(1 for v in context_data.values() if v)} of {len(context_data)} data sources")

Chat Orchestrator: all agent outputs loaded into context
Loaded 11 of 11 data sources


In [2]:
INTENT_CATEGORIES = {
    "readiness": "questions about career readiness score, overall progress, weak areas",
    "ats": "questions about resume ATS score, resume quality",
    "skills": "questions about what skills the candidate has",
    "skill_gap": "questions about missing skills, skill gap for a target role",
    "roadmap": "questions about learning plan, what to learn next, timeline",
    "projects": "questions about project ideas, what projects to build",
    "certifications": "questions about certifications to pursue",
    "resume_improvement": "questions about how to improve the resume, resume feedback",
    "interview": "questions about interview preparation, interview questions",
    "crafted_resume": "questions about the AI-rewritten/improved resume version",
    "jobs": "questions about job openings, job matches, applying to jobs",
    "general": "general greeting, unclear intent, or anything not covered above"
}

def classify_intent(user_message):
    categories_desc = "\n".join([f"- {k}: {v}" for k, v in INTENT_CATEGORIES.items()])
    prompt = f"""Classify this user question into exactly ONE category from the list below. Respond with ONLY the category name, nothing else.

Categories:
{categories_desc}

User question: "{user_message}"

Category:"""

    result = call_llm(prompt, temperature=0.1, max_tokens=20)
    result_clean = result.strip().lower().split()[0] if result.strip() else "general"

    for category in INTENT_CATEGORIES:
        if category in result_clean:
            return category
    return "general"

# Quick test
test_intent = classify_intent("What should I learn next to become an AI Engineer?")
print(f"Test classification: '{test_intent}'")

Test classification: 'roadmap'


In [3]:
def build_context_snippet(intent, data):
    if intent == "readiness":
        return json.dumps(data["readiness"].get("readiness", {}), indent=2)
    elif intent == "ats":
        return json.dumps(data["readiness"].get("ats", {}), indent=2)
    elif intent == "skills":
        return json.dumps(data["skills"].get("llm_based", {}), indent=2)
    elif intent == "skill_gap":
        return json.dumps(data["skill_gap"], indent=2)
    elif intent == "roadmap":
        return json.dumps(data["roadmap"], indent=2)
    elif intent == "projects":
        return json.dumps(data["projects"], indent=2)[:1500]
    elif intent == "certifications":
        return json.dumps(data["certifications"], indent=2)
    elif intent == "resume_improvement":
        return json.dumps(data["improvements"], indent=2)[:1500]
    elif intent == "interview":
        return json.dumps(data["interview"], indent=2)[:1500]
    elif intent == "crafted_resume":
        return data["crafted_resume"][:1500]
    elif intent == "jobs":
        return json.dumps(data["job_matches"], indent=2)[:1500]
    else:
        return json.dumps(data["readiness"].get("readiness", {}), indent=2)

def chat_respond(user_message, data=context_data):
    intent = classify_intent(user_message)
    context_snippet = build_context_snippet(intent, data)

    prompt = f"""You are CareerForge AI, an autonomous career mentor for college students preparing for placements.
You are warm, encouraging, and specific. Answer the student's question using ONLY the data provided below.
Do not make up information not present in the data. Keep the answer conversational and under 150 words.

RELEVANT DATA (category: {intent}):
{context_snippet}

STUDENT QUESTION: {user_message}

Your response:"""

    response = call_llm(prompt, temperature=0.6, max_tokens=350)
    return {"intent": intent, "response": response}

print("Chat response engine ready")

Chat response engine ready


In [4]:
test_questions = [
    "What is my career readiness score and what does it mean?",
    "What should I learn next?",
    "How can I improve my resume?",
    "What interview questions should I prepare for?",
    "Are there any jobs that match my skills?"
]

for q in test_questions:
    print(f"USER: {q}")
    result = chat_respond(q)
    print(f"[Intent detected: {result['intent']}]")
    print(f"CAREERFORGE AI: {result['response']}\n")
    print("-" * 70)

USER: What is my career readiness score and what does it mean?
[Intent detected: readiness]
CAREERFORGE AI: Hey there! Your current CareerForge AI readiness score is **39.0 out of 100**. While that might feel low right now, let's break down what this actually means by looking at your strengths versus the areas needing attention.

The good news? You're a total project enthusiast with a perfect **100** in projects and an impressive **95%** on your resume! Your ability to build things is clearly shining through. However, that 39% score reflects zeros across four critical foundational skills: Data Structures & Algorithms (DSA), Communication, Interview preparation, and Consistency.

Essentially, while you have the passion to create solutions, we need to strengthen how those ideas are structured logically during problem-solving sessions and how clearly you express them under pressure. Think of this not as a failure, but as your personalized roadmap highlighting exactly where the next 10% co

In [5]:
def start_chat():
    print("=" * 60)
    print("CareerForge AI — Chat with your career mentor")
    print("(Type 'exit' to end the conversation)")
    print("=" * 60)

    chat_history = []
    while True:
        user_input = input("\nYou: ")
        if user_input.lower() in ["exit", "quit", "bye"]:
            print("CareerForge AI: Good luck with your placement journey! 🚀")
            break

        result = chat_respond(user_input)
        print(f"\nCareerForge AI: {result['response']}")
        chat_history.append({"user": user_input, "intent": result["intent"], "response": result["response"]})

    return chat_history

# Uncomment to run interactively:
# history = start_chat()

print("Chat interface ready. Run start_chat() in a new cell to begin an interactive session.")

Chat interface ready. Run start_chat() in a new cell to begin an interactive session.


In [7]:
sample_conversation = []
for q in test_questions:
    result = chat_respond(q)
    sample_conversation.append({"user": q, "intent": result["intent"], "assistant": result["response"]})

with open("../outputs/sample_chat_log.json", "w", encoding="utf-8") as f:
    json.dump(sample_conversation, f, indent=2)

print("Sample conversation saved to ../outputs/sample_chat_log.json")
print("Notebook 16 (Chat Orchestrator Agent) — COMPLETE")
print("\nThis demonstrates all 15 agents accessible through one conversational interface,")
print("with intent-based routing — the core of an autonomous agentic system.")

Sample conversation saved to ../outputs/sample_chat_log.json
Notebook 16 (Chat Orchestrator Agent) — COMPLETE

This demonstrates all 15 agents accessible through one conversational interface,
with intent-based routing — the core of an autonomous agentic system.
